In [1]:
from pathlib import Path

import pandas as pd
import uproot

from utils import TREE_NAME, branch_summary, sample_label

pd.set_option("display.max_columns", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_DIR = PROJECT_ROOT / "output" / "florian"
INPUT_FILES = [
    INPUT_DIR / "ZKK.root",
    INPUT_DIR / "Zmumu.root",
    INPUT_DIR / "Zpipi.root",
]
PLOTS_DIR = PROJECT_ROOT / "plots" / "florian"


def _contains_label(labels: str, label: str) -> bool:
    return label in labels.split(", ") if labels else False


def sanity_summary(paths: list[Path], branch_table: pd.DataFrame) -> pd.DataFrame:
    branches = branch_table["branch"].astype(str)
    rows = []

    for path in paths:
        tree = uproot.open(path)[TREE_NAME]
        file_name = sample_label(path)
        missing = branch_table[
            branch_table["missing_files"].apply(lambda labels: _contains_label(labels, file_name))
        ]
        empty = branch_table[
            branch_table["empty_files"].apply(lambda labels: _contains_label(labels, file_name))
        ]

        rows.append(
            {
                "file": file_name,
                "sample": file_name,
                "entries": tree.num_entries,
                "branches_in_file": len(tree.keys()),
                "summary_branches": len(branch_table),
                "sdst_branches": int(branches.str.startswith("SDST_").sum()),
                "raw_sdst_branches": int(branches.str.startswith("RAWSDST_").sum()),
                "raw_fadana_branches": int(branches.str.startswith("RAWFADANA_").sum()),
                "present_branches": len(branch_table) - len(missing),
                "missing_branch_count": len(missing),
                "missing_branches": ", ".join(missing["branch"]),
                "empty_branch_count": len(empty),
                "empty_branches": ", ".join(empty["branch"]),
                "numeric_branches": int(branch_table["is_numeric"].sum()),
                "nan_values": int(branch_table["nan_count"].sum()),
            }
        )

    return pd.DataFrame(rows)


files = [Path(path) for path in INPUT_FILES]
missing_files = [path for path in files if not path.exists()]
assert not missing_files, "Missing input ROOT files:\n" + "\n".join(str(path) for path in missing_files)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Found {len(files)} ROOT files")
print(f"Writing plots and tables to {PLOTS_DIR}")


Found 3 ROOT files
Writing plots and tables to /eos/home-j/joshin/workspace-eos/delphi/delphi-nanoaod/plots/florian


In [2]:
branch_table = branch_summary(files)
summary = sanity_summary(files, branch_table)

summary.to_csv(PLOTS_DIR / "sanity_summary.csv", index=False)
display(summary)


,file,sample,entries,branches_in_file,summary_branches,sdst_branches,raw_sdst_branches,raw_fadana_branches,present_branches,missing_branch_count,missing_branches,empty_branch_count,empty_branches,numeric_branches,nan_values
0,ZKK,ZKK,1800,523,523,273,124,124,523,0,,78,"RAWFADANA_ElidRaw_gammaConvTag, RAWFADANA_Elid...",445,0
1,Zmumu,Zmumu,1800,523,523,273,124,124,523,0,,78,"RAWFADANA_ElidRaw_gammaConvTag, RAWFADANA_Elid...",445,0
2,Zpipi,Zpipi,1800,523,523,273,124,124,523,0,,78,"RAWFADANA_ElidRaw_gammaConvTag, RAWFADANA_Elid...",445,0


In [3]:
import awkward as ak
import numpy as np

META_TREE_NAME = "Meta"
META_SOURCES = ("sdst", "raw_sdst", "raw_fadana")
META_VECTOR_SUFFIXES = ("run_number", "event_number", "event_valid", "event_merged")


def _meta_numpy(array) -> np.ndarray:
    return np.asarray(ak.to_numpy(array), dtype=np.int64)


def meta_sanity_summary(paths: list[Path]) -> tuple[pd.DataFrame, list[str]]:
    required_branches = {
        f"n_{source}_{count}"
        for source in META_SOURCES
        for count in ("input", "valid", "merged")
    } | {
        f"{source}_{suffix}"
        for source in META_SOURCES
        for suffix in META_VECTOR_SUFFIXES
    }
    rows = []
    problems = []

    for path in paths:
        file_name = sample_label(path)
        with uproot.open(path) as root_file:
            if META_TREE_NAME not in root_file:
                problems.append(f"{file_name}: missing {META_TREE_NAME} tree")
                continue

            events = root_file[TREE_NAME]
            meta = root_file[META_TREE_NAME]
            missing = sorted(required_branches - set(meta.keys()))
            if missing:
                missing_text = ", ".join(missing)
                problems.append(f"{file_name}: missing Meta branches: {missing_text}")
                continue
            if meta.num_entries == 0:
                problems.append(f"{file_name}: Meta tree has no entries")

            arrays = meta.arrays(sorted(required_branches), library="ak", how=dict)
            merged_counts = {}

            for source in META_SOURCES:
                n_input = _meta_numpy(arrays[f"n_{source}_input"])
                n_valid = _meta_numpy(arrays[f"n_{source}_valid"])
                n_merged = _meta_numpy(arrays[f"n_{source}_merged"])
                valid_flags = arrays[f"{source}_event_valid"]
                merged_flags = arrays[f"{source}_event_merged"]

                if np.any((n_input < n_valid) | (n_valid < n_merged) | (n_merged < 0)):
                    problems.append(f"{file_name}/{source}: expected input >= valid >= merged >= 0")

                for suffix in META_VECTOR_SUFFIXES:
                    lengths = _meta_numpy(ak.num(arrays[f"{source}_{suffix}"], axis=1))
                    if not np.array_equal(lengths, n_input):
                        problems.append(f"{file_name}/{source}: {suffix} lengths do not match n_{source}_input")

                if not np.array_equal(_meta_numpy(ak.sum(valid_flags, axis=1)), n_valid):
                    problems.append(f"{file_name}/{source}: valid flag sums do not match n_{source}_valid")
                if not np.array_equal(_meta_numpy(ak.sum(merged_flags, axis=1)), n_merged):
                    problems.append(f"{file_name}/{source}: merged flag sums do not match n_{source}_merged")
                if bool(ak.any(merged_flags & ~valid_flags)):
                    problems.append(f"{file_name}/{source}: a merged event is marked invalid")

                merged_counts[source] = n_merged
                rows.append(
                    {
                        "file": file_name,
                        "source": source,
                        "events": events.num_entries,
                        "meta_entries": meta.num_entries,
                        "input": int(n_input.sum()),
                        "valid": int(n_valid.sum()),
                        "invalid": int((n_input - n_valid).sum()),
                        "merged": int(n_merged.sum()),
                        "valid_not_merged": int((n_valid - n_merged).sum()),
                    }
                )

            reference_merged = merged_counts[META_SOURCES[0]]
            if any(not np.array_equal(merged_counts[source], reference_merged) for source in META_SOURCES[1:]):
                problems.append(f"{file_name}: per-job merged counts differ between sources")
            if any(int(counts.sum()) != events.num_entries for counts in merged_counts.values()):
                problems.append(f"{file_name}: total merged counts do not match Events entries")

            for meta_suffix, event_branch in (("event_number", "Event_eventNumber"), ("run_number", "Event_runNumber")):
                if event_branch not in events.keys():
                    continue
                event_values = events[event_branch].array(library="np")
                meta_values = ak.to_numpy(
                    ak.flatten(arrays[f"sdst_{meta_suffix}"][arrays["sdst_event_merged"]], axis=None)
                )
                if not np.array_equal(meta_values, event_values):
                    problems.append(f"{file_name}: Events.{event_branch} does not match merged SDST metadata")

    return pd.DataFrame(rows), problems


In [4]:
meta_summary, meta_problems = meta_sanity_summary(files)

meta_summary.to_csv(PLOTS_DIR / "meta_sanity_summary.csv", index=False)
display(meta_summary)

assert not meta_problems, "Meta sanity check failed:\n" + "\n".join(f"- {problem}" for problem in meta_problems)
print(f"Meta sanity checks passed for {len(files)} ROOT files")


,file,source,events,meta_entries,input,valid,invalid,merged,valid_not_merged
0,ZKK,sdst,1800,4,1800,1800,0,1800,0
1,ZKK,raw_sdst,1800,4,1804,1800,4,1800,0
2,ZKK,raw_fadana,1800,4,1808,1800,8,1800,0
3,Zmumu,sdst,1800,4,1800,1800,0,1800,0
4,Zmumu,raw_sdst,1800,4,1804,1800,4,1800,0
5,Zmumu,raw_fadana,1800,4,1808,1800,8,1800,0
6,Zpipi,sdst,1800,4,1800,1800,0,1800,0
7,Zpipi,raw_sdst,1800,4,1804,1800,4,1800,0
8,Zpipi,raw_fadana,1800,4,1808,1800,8,1800,0


Meta sanity checks passed for 3 ROOT files
